# Нейронные сети и обработка естественного языка - NLP

# Модуль 5. Дообучение языковых моделей

### HF-токен

**ВНИМАНИЕ**: Вам понадобится `HF_TOKEN` для загрузки некоторых моделей и использования API.

Как получить токен:

1. Зарегистрироваться на Hugging Face: https://huggingface.co/join
2. В меню по клику на аватар выбрать пункт [Access Tokens](https://huggingface.co/settings/tokens)

3. Нажать `New token`
4. Выбрать:

   * `read` — достаточно почти для всех курсов и скачивания моделей
5. Скопировать токен вида:

    ```text
    hf_xxxxxxxxxxxxxxxxx
    ```

Полученный токен следует разместить в файле ```__config__.py``` в виде переменной `HF_TOKEN`:

```python
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxx"
```

In [ ]:
import os
import numpy as np
import torch

from datasets import Dataset, load_dataset

from __config__ import *

print(len(HF_TOKEN))


os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HOME"] = "./hf_cache" # для колаба ок

## 1. Trainer

Модуль обучения моделей трансформеров с помощью `Trainer` и кастомных циклов обучения.

Работает как для обучения с нуля, так и для дообучения предобученных моделей на своих данных.

Поддерживает распределенное обучение, смешанные типы данных и различные оптимизаторы.

In [ ]:
from transformers import (
    AutoTokenizer,
    GPT2Config,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    pipeline
)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

### 1.1 Обучение трансформера с нуля

In [ ]:
proverbs = [
    "Без труда не вытащишь и рыбку из пруда.",
    "Семь раз отмерь, один раз отрежь.",
    "Тише едешь, дальше будешь.",
    "Любишь кататься, люби и саночки возить.",
    "Делу время, потехе час.",
    "Не имей сто рублей, а имей сто друзей.",
    "Слово не воробей, вылетит не поймаешь.",
    "Яблоко от яблони недалеко падает.",
    "Век живи, век учись.",
    "Что посеешь, то и пожнёшь.",
    "Один в поле не воин.",
    "Лучше поздно, чем никогда.",
    "Не всё то золото, что блестит.",
    "Старый друг лучше новых двух.",
    "Под лежачий камень вода не течёт.",
    "Готовь сани летом, а телегу зимой.",
    "Цыплят по осени считают.",
    "Дарёному коню в зубы не смотрят.",
    "За двумя зайцами погонишься, ни одного не поймаешь.",
    "В гостях хорошо, а дома лучше.",
    "Кто рано встаёт, тому Бог подаёт.",
    "Москва не сразу строилась.",
    "Не рой другому яму, сам в неё попадёшь.",
    "Худой мир лучше доброй ссоры.",
    "Лес рубят, щепки летят.",
    "Аппетит приходит во время еды.",
    "Кашу маслом не испортишь.",
    "После драки кулаками не машут.",
    "На безрыбье и рак рыба.",
    "Не зная броду, не суйся в воду.",
] * 100

dataset = Dataset.from_dict({"text": proverbs})
dataset

In [ ]:
tokenizer_name = "sberbank-ai/rugpt3small_based_on_gpt2"

tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=48,
    )

tokenized = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

tokenized

In [ ]:
# модель
config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=48,
    n_ctx=48,
    n_embd=128,
    n_layer=2,
    n_head=4,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

model = GPT2LMHeadModel(config).to(device)

print("Параметров:", round(sum(p.numel() for p in model.parameters()) / 1e6, 2), "млн")

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False # для языковых моделей отключаем маскирование
)

training_args = TrainingArguments(
    output_dir="./russian-proverbs-gpt-demo",
    num_train_epochs=20,
    per_device_train_batch_size=16,
    learning_rate=5e-4,
    logging_steps=10,
    save_steps=500,
    save_total_limit=1,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

result = generator(
    "Без труда",
    max_new_tokens=12,
    do_sample=True,
    temperature=0.2,
    top_k=10,
    pad_token_id=tokenizer.eos_token_id
)

print(result[0]["generated_text"])

In [ ]:
starts = [
    "Без труда",
    "Семь раз",
    "Тише едешь",
    "Слово не",
    "За двумя зайцами",
    "Не всё то",
]

for s in starts:
    print("=" * 60)
    print(generator(
        s,
        max_new_tokens=12,
        do_sample=True,
        temperature=0.7,
        top_k=30,
        pad_token_id=tokenizer.eos_token_id
    )[0]["generated_text"])

In [ ]:
# посмотреть 5 следующих токенов
def predict_next_tokens(prompt, top_k=5):
    model.eval()

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[0, -1]
    probs = torch.softmax(logits, dim=-1)

    top = torch.topk(probs, top_k)

    for token_id, prob in zip(top.indices, top.values):
        token = tokenizer.decode([token_id.item()])
        print(f"{repr(token):15s} {prob.item():.8f}")

predict_next_tokens("Без труда", top_k=10)

### 1.2 Дообучение предобученной модели

In [ ]:
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
positive_texts = [
    "Отличный курс, всё понятно и интересно.",
    "Материал объяснён хорошо, примеры полезные.",
    "Очень понравилась подача преподавателя.",
    "Курс оказался полезным и практичным.",
    "Прекрасное объяснение сложной темы.",
    "Мне понравились задания и примеры.",
    "Отличная лекция, многое стало ясно.",
]

negative_texts = [
    "Курс скучный, ничего не понял.",
    "Материал подан плохо и запутанно.",
    "Примеры бесполезные, объяснение слабое.",
    "Мне не понравилась структура занятия.",
    "Слишком сложно и непонятно.",
    "Лекция была затянута и неинтересна.",
    "Ожидал большего, курс разочаровал.",
]

texts = (positive_texts + negative_texts) * 20
labels = ([1] * len(positive_texts) + [0] * len(negative_texts)) * 20

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
test_ds = Dataset.from_dict({"text": test_texts, "label": test_labels})

In [ ]:
model_name = "DeepPavlov/rubert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

id2label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1
}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

train_tok = train_ds.map(tokenize, batched=True)
test_tok = test_ds.map(tokenize, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./rubert-sentiment-demo",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
)

examples = [
    "Очень полезный и понятный курс.",
    "Совершенно бесполезная лекция, ничего не понял.",
    "Примеры хорошие, но темп слишком быстрый.",
]

clf(examples)

### 1.3 SFT и обучение LoRA

- **SFT** (Supervised Fine-Tuning) — обучение модели формату поведения.
- **RLHF** (Reinforcement Learning with Human Feedback) — это метод дообучения языковых моделей, который использует обратную связь от человека.

- **LoRA** (Low-Rank Adaptation) — работает путем добавления низкоранговых адаптационных слоев к существующим слоям модели. В обучении используются только они.

Рассмотрим на примере дообучения модели чат-бота скалодрома на диалогах в стиле былины. В этом примере мы будем использовать SFT для обучения модели отвечать в стиле былины:

    - Я первый раз, мне страшно. Подскажите, пожалуйста.
    - Не бойся, добрый человек. Первый путь будет не богатырским испытанием, а спокойным знакомством: наставник покажет, как двигаться безопасно.


In [ ]:
!pip install peft trl

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig # PEFT = Parameter-Efficient Fine-Tuning.
from trl import SFTTrainer, SFTConfig # TRL = Transformer Reinforcement Learning

In [ ]:
bylina = load_dataset("json", data_files="https://github.com/easyise/spec_python_courses/raw/refs/heads/master/neural_02_nlp/data/sft_bylina_climbing_200.json")
bylina

In [ ]:
bylina['train'][1]

In [ ]:
dataset = bylina['train']
dataset[:5]

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_base = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

In [ ]:
messages = [
    {"role": "system", "content": "Ты чат-бот-сказитель. Отвечай в стиле русских былин: торжественно, образно, но понятно."},
    {"role": "user", "content": "Как отменить запись на тренировку?"}
]

text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

print("Промпт до обучения:")
print(text)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model_base.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

print("Ответ до обучения:")
print(tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:],
                        skip_special_tokens=True))

In [ ]:
# специальная подготовка данных для обучения чат-бота
def format_chat(example):

    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

dataset = dataset.map(format_chat)

In [ ]:
print(dataset[0]['text'])

In [ ]:
# настройки LoRA
peft_config = LoraConfig(
    r=8, # ранг разложения матриц весов, чем меньше r, тем меньше обучаемых параметров и ниже качество
    lora_alpha=16, # коэффициент масштабирования LoRA-адаптеров, помогает стабилизировать обучение при малом r
    lora_dropout=0.05, # вероятность отключения адаптеров во время обучения для регуляризации
    bias="none", # обучаемые параметры не должны включать смещения, так как они могут быть критичными для модели и их изменение может привести к ухудшению производительности
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

In [ ]:
# настройки SFT-тренера
sft_config = SFTConfig(
    output_dir="./bylina-style-chatbot",
    dataset_text_field="text",
    max_length=256,
    packing=False,

    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-5,
    warmup_steps=10,
    max_grad_norm=0.3,

    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

In [ ]:
model.config.use_cache = False

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    peft_config=peft_config,
)

trainer.train()

In [ ]:
trained_model = trainer.model
trained_model.eval()

In [ ]:
messages = [
    {"role": "system", "content": "Ты чат-бот-сказитель. Отвечай в стиле русских былин: торжественно, образно, но понятно. Не выдумывай точные цены и расписание, если их нет в вопросе; предлагай уточнить у администратора."},
    {"role": "user", "content": "Нужна ли специальная обувь?"}
]

text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

print("Промпт после обучения:")
print(text)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = trained_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.3,
        top_k=30,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print("Ответ после обучения:")
print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:],
                        skip_special_tokens=True))

In [ ]:
with trainer.model.disable_adapter():
    output_ids = trained_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.3,
        top_k=30,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print("Ответ без адаптера:")
print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:],
                        skip_special_tokens=True))

In [ ]:
trainer.model.print_trainable_parameters()

**ПРАКТИКА**

1. Попробуйте варьировать параметры генерации, включая формулировку вопросов чат-бота.

2. Попробуйте изменить настройки тренера и LoRA, чтобы добиться лучшего качества. 

3. Попробуйте использовать другие модели, например
```
        Vikhrmodels/Vikhr-Qwen-2.5-1.5B-Instruct
        Qwen/Qwen2.5-1.5B-Instruct
        Qwen/Qwen2.5-3B-Instruct
```

4. Попробуйте дообучить модель на другом наборе данных, например https://github.com/easyise/spec_python_courses/blob/master/neural_02_nlp/data/sft_imperial_russia_style_200.json

In [ ]:
# ваш код здесь







### 1.4 RLHF и DPO

RLHF (Reinforcement Learning with Human Feedback) — это метод дообучения языковых моделей, который использует обратную связь от человека для улучшения качества генерации. В рамках RLHF модель обучается на основе предпочтений человека, которые могут быть выражены в виде оценок или сравнений между различными ответами модели.

**PPO** (Proximal Policy Optimization) — в основе этой модели лежит использование баллов вознаграждения (**reward**), которые передаются модели **Reward Model**, которая обучается вместе с LLM, стремясь максимизировать **reward**.

**DPO** (Direct Preference Optimization) — здесь в основе лежит триплет `(prompt, chosen, rejected)`, где `prompt` — это исходный запрос, `chosen` — предпочтительный ответ, а `rejected` — менее предпочтительный ответ. Модель при этом выполняет оптимизацию, стремясь увеличить вероятность генерации `chosen` и уменьшить вероятность генерации `rejected`.

| PPO                | DPO              |
| ------------------ | ---------------- |
| RL                 | supervised-like  |
| нужна reward model | не нужен         |
| sampling loop      | обычный training |
| сложно             | проще            |
| дорого             | дешевле          |
| нестабильно        | стабильно        |


In [ ]:
# пример данных
raw_data = [
    {
        "user": "Как отменить запись на тренировку?",
        "chosen": "Ой ты гой еси, добрый человек! Коли запись отменить желаешь, напиши администратору заранее, и он перенесёт или отменит твоё занятие без лишней суеты.",
        "rejected": "Напишите администратору для отмены записи."
    },
    {
        "user": "Нужна ли специальная обувь?",
        "chosen": "Ой ты гой еси! Для лазания нужны скальные туфли: они крепче держат ногу на зацепах. Коли своих нет, не кручинься — их можно взять в прокат.",
        "rejected": "Да, нужна специальная обувь. Можно взять в прокат."
    },
    {
        "user": "Я первый раз, мне можно прийти?",
        "chosen": "Можно, добрый молодец! Для первого пути подойдёт вводное занятие: инструктор покажет правила, снаряжение и простые трассы.",
        "rejected": "Да, новички могут прийти на вводное занятие."
    },
    {
        "user": "Что взять с собой?",
        "chosen": "Возьми, душа светлая, одежду удобную, воды студёной да настроение бодрое. А снаряжение, коли своего нет, найдётся в прокате.",
        "rejected": "Возьмите спортивную одежду и воду."
    },
] * 50

In [ ]:
system_prompt = (
    "Ты чат-бот-сказитель. Отвечай в стиле русских былин: "
    "торжественно, образно, но понятно."
)

def make_dpo_example(example):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": example["user"]},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return {
        "prompt": prompt,
        "chosen": example["chosen"] + tokenizer.eos_token,
        "rejected": example["rejected"] + tokenizer.eos_token,
    }

dataset = Dataset.from_list(raw_data).map(make_dpo_example)
dataset = dataset.remove_columns(["user"])

print(dataset[0]["prompt"])
print("CHOSEN:", dataset[0]["chosen"])
print("REJECTED:", dataset[0]["rejected"])

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
)

model.config.use_cache = False

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir="./dpo-bylina-demo",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    num_train_epochs=3,

    learning_rate=2e-5,
    warmup_steps=5,

    max_grad_norm=0.3,

    max_length=256,

    beta=0.1,

    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

In [ ]:
trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()

In [ ]:
def generate_answer(model_to_use, question):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model_to_use.device)

    with torch.no_grad():
        output_ids = model_to_use.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

print(generate_answer(trainer.model, "Нужна ли специальная обувь?"))

In [ ]:
print("С DPO-адаптером:")
print(generate_answer(trainer.model, "Как отменить запись?"))

print("\nБез адаптера:")
with trainer.model.disable_adapter():
    print(generate_answer(trainer.model, "Как отменить запись?"))

## 2. Дообучение и разметка данных

Мы уже познакомились с разметкой данных для обучения чат-ботов и моделей для классификации в примерах в разделе 1 настроящего ноутбука. 

### 2.1 Разметка для NER

```
слово → метка
```

Пример:

```
Москва → B-LOC
столица → O
России → B-LOC
```

BIO-разметка:
```
B-LOC — начало сущности типа "место"
I-LOC — продолжение сущности типа "место"
O — слово не является частью сущности
```

Пример:
```
Великий → B-LOC
Новгород → I-LOC
- → O
столица → O
России → B-LOC
```

Пример ручной разметки для NER:

```python
tokens = ["Илон", "Маск", "приехал", "в", "Москву"]
ner_tags = ["B-PER", "I-PER", "O", "O", "B-LOC"]
```


In [ ]:
!pip install -q evaluate seqeval accelerate

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

import evaluate

In [ ]:
dataset = load_dataset("unimelb-nlp/wikiann", "ru")
dataset

In [ ]:
dataset["train"][42]

In [ ]:
# что размечено в данных
features = dataset["train"].features
label_names = features["ner_tags"].feature.names

label_names

In [ ]:
# что к чему
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

id2label

In [ ]:
model_name = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

#### проблема выравнивания токенов и меток

исходные слова:
`["Международная", "организация"]`

после tokenizer:
`["Международ", "##ная", "организация"]`

In [ ]:
# Внимание на комментарии в коде
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = examples["ner_tags"]
    new_labels = []

    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100) # -100 — специальное значение, которое игнорируется при вычислении потерь

            elif word_id != previous_word_id:
                label_ids.append(labels[word_id]) # для первого токена слова используем его метку

            else:
                label_ids.append(-100) # для остальных токенов слова используем -100, чтобы игнорировать их при обучении

            previous_word_id = word_id

        new_labels.append(label_ids)

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

In [ ]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

In [ ]:
small_train = tokenized_dataset["train"].shuffle(seed=42).select(range(3000))
small_valid = tokenized_dataset["validation"].shuffle(seed=42).select(range(500))
small_test = tokenized_dataset["test"].shuffle(seed=42).select(range(500))

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
# коллатор для токен-классификации: добавляет паддинг и формирует батчи
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [ ]:
# для снятия метрик используем evaluate и seqeval
# seqeval считает композицию из accuracy, precision, recall и F1 для задач разметки последовательностей
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        current_preds = []
        current_labels = []

        for p, l in zip(pred, lab):
            if l != -100:
                current_preds.append(label_names[p])
                current_labels.append(label_names[l])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./wikiann-ru-ner-demo",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
trainer.evaluate(small_test)

In [ ]:
ner = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

In [ ]:
text = "Илон Маск встретился с представителями Яндекса в Москве."

ner(text)

**ПРАКТИКА** 

1. Попробуйте варьировать параметры обучения и пайплайна. Сделайте выводы.
2. Выполните ручную разметку для NER:

In [ ]:
manual_examples = [
    {
        "tokens": ["Анна", "Каренина", "жила", "в", "Петербурге"],
        "ner_tags": ["B-PER", "I-PER", "O", "O", "B-LOC"]
    },
    {
        "tokens": ["Сбербанк", "открыл", "офис", "в", "Казани"],
        "ner_tags": ???
    },
    {
        "tokens": ["Лев", "Толстой", "написал", "Войну", "и", "мир"],
        "ner_tags": ???
    }
]

### 2.2 Общие вопросы разметки и дообучения

Подводим итоги.

**Вопрос**: как размечать данные для решения следующих задач:
1. Классификация текста
2. Анализ тональности (позитив/негатив)
3. Автодополнение и исправление текста
4. Машинный перевод
5. Извлечение именованных сущностей и информации
6. Поиск похожих текстов
7. Семантический поиск
8. Генерация текста
9. Суммаризация текста
10. Ответы на вопросы по тексту
11. Диалоговые системы
12. Поиск аномалий в текстах
